# Агентный SIR

В агентной постановке каждый агент хранит своё состояние, длительность болезни и текущий город. На каждом шаге моделирования учитываются заражения внутри города, выздоровление и миграция.

In [ ]:
using Random
using Printf

results_dir = normpath(joinpath(@__DIR__, "..", "results", "data"))
mkpath(results_dir)
Random.seed!(7)

mutable struct Agent
    state::Int
    city::Int
    days::Int
end

function write_csv(path, headers, rows)
    open(path, "w") do io
        println(io, join(headers, ","))
        for row in rows
            println(io, join(string.(row), ","))
        end
    end
end

function simulate_agent_sir(; migration = 0.04, steps = 120, beta = 0.28, infection_period = 10)
    agents = Agent[]
    for i in 1:180
        state = i <= 6 ? 2 : 1
        push!(agents, Agent(state, rand(1:3), 0))
    end
    rows = Vector{Vector{String}}()
    for step in 1:steps
        for city in 1:3
            ids = findall(a -> a.city == city, agents)
            n_city = max(length(ids), 1)
            infected = count(i -> agents[i].state == 2, ids)
            for idx in ids
                if agents[idx].state == 1
                    p = 1 - exp(-beta * infected / n_city)
                    if rand() < p
                        agents[idx].state = 2
                        agents[idx].days = 0
                    end
                end
            end
        end
        for agent in agents
            if agent.state == 2
                agent.days += 1
                if agent.days >= infection_period
                    agent.state = 3
                end
            end
            if rand() < migration
                target = rand(1:3)
                agent.city = target
            end
        end
        s = count(a -> a.state == 1, agents)
        i = count(a -> a.state == 2, agents)
        r = count(a -> a.state == 3, agents)
        push!(rows, [string(step), string(s), string(i), string(r)])
    end
    return rows
end

main_rows = simulate_agent_sir()
write_csv(joinpath(results_dir, "agent_sir.csv"), ["step", "S", "I", "R"], main_rows)

sweep_rows = Vector{Vector{String}}()
for migration in 0.00:0.05:0.20
    rows = simulate_agent_sir(migration = migration)
    peaks = maximum(parse.(Float64, getindex.(rows, 3)))
    push!(sweep_rows, [@sprintf("%.2f", migration), @sprintf("%.3f", peaks)])
end
write_csv(joinpath(results_dir, "migration_sweep.csv"), ["migration", "peak_infected"], sweep_rows)
println("lab04 done")

Сохранённые результаты позволяют сравнить основной сценарий с серией прогонов по интенсивности миграции.